In [ ]:
import pandas as pd
import requests

In [ ]:
def get_switchy_links(api_key: str) -> pd.DataFrame:
    """
    Envia uma query GraphQL para o endpoint do Switchy.io e retorna um DataFrame com os links.

    Args:
        api_key (str): Chave da API Switchy (valor do header Api-Authorization)

    Returns:
        pd.DataFrame: Contendo os campos id e url dos links
    """
    url = "https://graphql.switchy.io/v1/graphql"
    
    headers = {
        "Content-Type": "application/json",
        "Api-Authorization": api_key
    }
    
    query = """
    query MyQuery {
        links {
            id
            url
        }
    }
    """
    
    response = requests.post(url, headers=headers, json={"query": query})
    
    # Verifica erros
    if response.status_code != 200:
        raise Exception(f"Erro na requisição: {response.status_code} - {response.text}")
    
    data = response.json()
    
    # Verifica erros GraphQL
    if "errors" in data:
        raise Exception(f"Erro na resposta GraphQL: {data['errors']}")
    
    # Extrai dados e converte para DataFrame
    links = data["data"]["links"]
    df = pd.DataFrame(links)
    
    return df

In [ ]:
def update_switchy_link_rotator(api_key: str, link_id: str, url_v2: str, url_v3: str, url_v4: str):
    """
    Atualiza o campo extraOptionsLinkRotator de um link no Switchy.io via GraphQL.

    Args:
        api_key (str): Chave de API do Switchy (header Api-Authorization)
        link_id (str): ID do link a ser atualizado
        url_v2 (str): Primeira URL do rotator
        url_v3 (str): Segunda URL do rotator

    Returns:
        dict: Resultado da mutation (affected_rows e dados retornados)
    """
    endpoint = "https://graphql.switchy.io/v1/graphql"

    headers = {
        "Content-Type": "application/json",
        "Api-Authorization": api_key
    }

    query = f"""
    mutation {{
      update_links(
        where: {{id: {{_eq: "{link_id}"}}}},
        _set: {{
          extraOptionsLinkRotator: [
            {{url: "{url_v2}", value: 25}},
            {{url: "{url_v3}", value: 25}},
            {{url: "{url_v4}", value: 25}}
          ]
        }}
      ) {{
        affected_rows
        returning {{
          id
          extraOptionsLinkRotator
        }}
      }}
    }}
    """

    response = requests.post(endpoint, headers=headers, json={"query": query})

    if response.status_code != 200:
        raise Exception(f"Erro HTTP {response.status_code}: {response.text}")

    data = response.json()

    if "errors" in data:
        raise Exception(f"Erro GraphQL: {data['errors']}")

    return data["data"]["update_links"]

In [ ]:
links = get_switchy_links('735c8051-ef80-4287-bd17-06d7176ad956	')

In [ ]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(links):
    # Filter rows based on column: 'url'
    links = links[links['url'].str.contains("pc-black-alunos-v1", regex=False, na=False, case=False)]
    links['utm_string'] = links['url'].apply(lambda x: x.split('?')[-1])
    links['url_v2'] = links['utm_string'].apply(lambda x: f'https://matheusjerke.com.br/pc-black-alunos-v2/?{x}')
    links['url_v3'] = links['utm_string'].apply(lambda x: f'https://matheusjerke.com.br/pc-black-alunos-v3/?{x}')
    links['url_v4'] = links['utm_string'].apply(lambda x: f'https://matheusjerke.com.br/pc-black-alunos-v4/?{x}')

    return links

links_clean = clean_data(links.copy())
links_clean.head()

In [ ]:
sample = links_clean.iloc[-1]


In [ ]:
update_switchy_link_rotator(
            api_key='735c8051-ef80-4287-bd17-06d7176ad956',
            link_id=sample.get('id'),
            url_v2=sample.get('url_v2'),
            url_v3=sample.get('url_v3'),
            url_v4=sample.get('url_v4')
        )

In [ ]:
for idx, row in links_clean.iterrows():
    print(f"Atualizando link {idx+1}/{len(links_clean)}: id={row.get('id')}")
    try:
        update_switchy_link_rotator(
            api_key='735c8051-ef80-4287-bd17-06d7176ad956',
            link_id=row.get('id'),
            url_v2=row.get('url_v2'),
            url_v3=row.get('url_v3'),
            url_v4=row.get('url_v4')
        )
        print(f"Link {row.get('id')} atualizado com sucesso.")
    except Exception as e:
        print(f"Erro ao atualizar o link {row.get('id')}: {e}")